# Sample images from LILA BC

1. Get image URLs and labels from LILA BC
2. Sample n images for each class
3. Train test split
4. Save CSV with sampled image URLs to Drive

## Setup

In [1]:
import os
import sys
import yaml
import subprocess
import numpy as np
import pandas as pd
import polars as pl
from google.colab import drive

In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# add folder to sys.path so that utilities can be imported
project_dir = 'drive/MyDrive/TeraiNet'
sys.path.append(project_dir)

In [4]:
from utilities import load_config, get_image_urls, sample_n_images_per_species, add_subset_column

In [5]:
# load config and set variables and parameters
config_path = os.path.join(project_dir, 'config.yaml')
config = load_config(config_path)

scripts_dir = os.path.join(project_dir, config['sampling_downloading']['scripts_dir'])
get_image_urls_script = os.path.join(scripts_dir, config['scripts']['get_image_urls_script'])

samples_dir = os.path.join(project_dir, config['sampling_downloading']['samples_dir'])

train_ratio = config['training']['train_ratio']

In [6]:
# for a fresh start, remove samples dir
remove_samples_dir = True
if remove_samples_dir:
  !rm -rf "$samples_dir"
  !mkdir -p "$samples_dir"

## Download image URLs and labels from LILA BC

In [7]:
!wget -O lila_image_urls_and_labels.csv.zip -nc "https://lila.science/public/lila_image_urls_and_labels.csv.zip"

File ‘lila_image_urls_and_labels.csv.zip’ already there; not retrieving.


In [8]:
![ -f lila_image_urls_and_labels.csv ] || unzip lila_image_urls_and_labels.csv.zip

In [9]:
urls_and_labels = 'lila_image_urls_and_labels.csv'

## Inspect species counts

Check taxonomy mapping to find relevant species:
https://lila.science/public/lila-taxonomy-mapping_release.csv

In [10]:
columns = [
    'url_gcp',
    'image_id',
    'sequence_id',
    'location_id',
    'frame_num',
    'datetime',
    'common_name'
]

schema_overrides = {
    'url_gcp': pl.Utf8(),
    'image_id': pl.Utf8(),
    'sequence_id': pl.Utf8(),
    'location_id': pl.Utf8(),
    'frame_num': pl.Int32(),
    'datetime': pl.Utf8(),
    'common_name': pl.Utf8()
}

lila_image_urls_and_labels_df = pl.read_csv(
    'lila_image_urls_and_labels.csv',
    columns=columns,
    schema_overrides=schema_overrides
)
lila_image_urls_and_labels_df.shape

(23723546, 7)

In [11]:
lila_image_urls_and_labels_df.head()

url_gcp,image_id,sequence_id,location_id,frame_num,common_name,datetime
str,str,str,str,i32,str,str
"""https://storage.googleapis.com…","""Caltech Camera Traps : 5968c0f…","""Caltech Camera Traps : 6f2160e…","""Caltech Camera Traps : 26""",1,null,"""10-04-2013 13:31:53"""
"""https://storage.googleapis.com…","""Caltech Camera Traps : 5a0b016…","""Caltech Camera Traps : 6f27ed6…","""Caltech Camera Traps : 26""",1,"""deer""","""11-04-2013 18:37:07"""
"""https://storage.googleapis.com…","""Caltech Camera Traps : 59b93af…","""Caltech Camera Traps : 6f04895…","""Caltech Camera Traps : 38""",2,"""cat""","""05-09-2012 07:33:45"""
"""https://storage.googleapis.com…","""Caltech Camera Traps : 59641f5…","""Caltech Camera Traps : 6f0385b…","""Caltech Camera Traps : 38""",2,"""virginia opossum""","""03-29-2012 02:34:13"""
"""https://storage.googleapis.com…","""Caltech Camera Traps : 5a1e530…","""Caltech Camera Traps : 6f0a3cc…","""Caltech Camera Traps : 33""",2,null,"""05-08-2012 19:23:36"""


In [12]:
# TODO: figure out why common_name is empty in half the rows and what this implies
lila_image_urls_and_labels_df.filter(pl.col('common_name').is_null()).shape

(11843789, 7)

In [13]:
species_list = [
    'tiger',
    'leopard',
    'asian black bear', 'american black bear', # not enough images of asian black bear alone
    'dhole', 'black-backed jackal', 'gray fox', 'leopard cat', 'mainland leopard cat', 'marbled cat', 'asian golden cat', # other carnivores (including substitutes, i. e. black-backed jackal and gray fox)
    'deer',
    'wild boar',
    'african buffalo', 'cape buffalo', # substitute for gaur
    'white rhinoceros', # substitute for indian rhino
    'asian elephant', 'african elephant', 'african bush elephant', # not enough images of asian elephant alone
    'bird'
]

In [14]:
lila_image_urls_and_labels_df = lila_image_urls_and_labels_df.filter(pl.col('common_name').is_in(species_list))
species_counts = lila_image_urls_and_labels_df['common_name'].value_counts()
pl.Config.set_tbl_rows(species_counts.shape[0])
species_counts.sort('count')

common_name,count
str,u32
"""dhole""",185
"""mainland leopard cat""",246
"""leopard cat""",266
"""marbled cat""",271
"""tiger""",321
"""asian elephant""",325
"""asian golden cat""",353
"""asian black bear""",1221
"""african elephant""",1434


## Get image URLs from LILA BC

The goal is to get about 3000 images per class (+100 buffer). Leopard images availability sets this limit because we want classes to be balanced. For tiger images, which are even scarcer, there fortunately is another additional source (Amur tiger re-identification challenge)

### Tiger

In [ ]:
class_name = 'tiger'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = 'tiger'
samples_file = get_image_urls(samples_dir, class_name, get_image_urls_script, urls_and_labels, column_to_filter, values_to_filter)

Getting image URLs based on common_name equal to any of {tiger}...
Total matching rows found: 321
Filtering complete. Result written to drive/MyDrive/TeraiNet/samples/lila_bc_image_urls_tiger.csv.




In [ ]:
# use all tiger images because we don't have many
species_samples_dict = {
    'tiger': 321,
}

sampled_image_urls = sample_n_images_per_species(samples_file, species_samples_dict, column_to_filter)
!rm "$samples_file"
sampled_image_urls['class_number'] = class_number
sampled_image_urls['subset'] = 'test2' # put all lila bc tiger images into separate test set to test how well model trained on amur tiger images generalizes
sampled_samples_file = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.to_csv(sampled_samples_file, index=False, sep=',')
!echo "Sampled rows: $(tail -n +2 "$sampled_samples_file" | wc -l)"
sampled_image_urls[column_to_filter].value_counts()

Sampled rows: 321


,count
common_name,
tiger,321


### Leopard

In [ ]:
class_name = 'leopard'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = 'leopard'
samples_file = get_image_urls(samples_dir, class_name, get_image_urls_script, urls_and_labels, column_to_filter, values_to_filter)

Getting image URLs based on common_name equal to any of {leopard}...
Total matching rows found: 2991
Filtering complete. Result written to drive/MyDrive/TeraiNet/samples/lila_bc_image_urls_leopard.csv.




In [ ]:
# use all leopard images because because we don't have many
species_samples_dict = {
    'leopard': 2991,
}

sampled_image_urls = sample_n_images_per_species(samples_file, species_samples_dict, column_to_filter)
!rm "$samples_file"
sampled_image_urls['class_number'] = class_number
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_samples_file = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.to_csv(sampled_samples_file, index=False, sep=',')
!echo "Sampled rows: $(tail -n +2 "$sampled_samples_file" | wc -l)"
sampled_image_urls[column_to_filter].value_counts()

Sampled rows: 2991


,count
common_name,
leopard,2991


### Black bear

Include `american black bear` because there are not enough camera trap images of Asian black bears.

In [ ]:
class_name = 'black_bear'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = 'asian black bear,american black bear'
samples_file = get_image_urls(samples_dir, class_name, get_image_urls_script, urls_and_labels, column_to_filter, values_to_filter)

Getting image URLs based on common_name equal to any of {asian black bear,american black bear}...
Total matching rows found: 34075
Filtering complete. Result written to drive/MyDrive/TeraiNet/samples/lila_bc_image_urls_black_bear.csv.




In [ ]:
species_samples_dict = {
    'asian black bear': 1221,
    'american black bear': 1879,
}

sampled_image_urls = sample_n_images_per_species(samples_file, species_samples_dict, column_to_filter)
!rm "$samples_file"
sampled_image_urls['class_number'] = class_number
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_samples_file = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.to_csv(sampled_samples_file, index=False, sep=',')
!echo "Sampled rows: $(tail -n +2 "$sampled_samples_file" | wc -l)"
sampled_image_urls[column_to_filter].value_counts()

Sampled rows: 3100


,count
common_name,
american black bear,1879
asian black bear,1221


### Other carnivores

Include `dhole,black-backed jackal,gray fox,leopard cat,mainland leopard cat,marbled cat,asian golden cat` to cover a wide range of other carnivores in the Terai ecosystem.

In [ ]:
class_name = 'other_carnivores'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = 'dhole,black-backed jackal,gray fox,leopard cat,mainland leopard cat,marbled cat,asian golden cat'
samples_file = get_image_urls(samples_dir, class_name, get_image_urls_script, urls_and_labels, column_to_filter, values_to_filter)

Getting image URLs based on common_name equal to any of {dhole,black-backed jackal,gray fox,leopard cat,mainland leopard cat,marbled cat,asian golden cat}...
Total matching rows found: 34402
Filtering complete. Result written to drive/MyDrive/TeraiNet/samples/lila_bc_image_urls_other_carnivores.csv.




In [ ]:
species_samples_dict = {
    'dhole': 185,
    'black-backed jackal': 890,
    'gray fox': 890,
    'leopard cat': 266,
    'mainland leopard cat': 246,
    'marbled cat': 271,
    'asian golden cat': 353,
}

sampled_image_urls = sample_n_images_per_species(samples_file, species_samples_dict, column_to_filter)
!rm "$samples_file"
sampled_image_urls['class_number'] = class_number
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_samples_file = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.to_csv(sampled_samples_file, index=False, sep=',')
!echo "Sampled rows: $(tail -n +2 "$sampled_samples_file" | wc -l)"
sampled_image_urls[column_to_filter].value_counts()

Sampled rows: 3101


,count
common_name,
black-backed jackal,890
gray fox,890
asian golden cat,353
marbled cat,271
leopard cat,266
mainland leopard cat,246
dhole,185


### Deer

In [ ]:
class_name = 'deer'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = 'deer'
samples_file = get_image_urls(samples_dir, class_name, get_image_urls_script, urls_and_labels, column_to_filter, values_to_filter)

Getting image URLs based on common_name equal to any of {deer}...
Total matching rows found: 360489
Filtering complete. Result written to drive/MyDrive/TeraiNet/samples/lila_bc_image_urls_deer.csv.




In [ ]:
species_samples_dict = {
    'deer': 3100,
}

sampled_image_urls = sample_n_images_per_species(samples_file, species_samples_dict, column_to_filter)
!rm "$samples_file"
sampled_image_urls['class_number'] = class_number
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_samples_file = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.to_csv(sampled_samples_file, index=False, sep=',')
!echo "Sampled rows: $(tail -n +2 "$sampled_samples_file" | wc -l)"
sampled_image_urls[column_to_filter].value_counts()

Sampled rows: 3100


,count
common_name,
deer,3100


### Wild boar

In [ ]:
class_name = 'wild_boar'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = 'wild boar'
samples_file = get_image_urls(samples_dir, class_name, get_image_urls_script, urls_and_labels, column_to_filter, values_to_filter)

Getting image URLs based on common_name equal to any of {wild boar}...
Total matching rows found: 142701
Filtering complete. Result written to drive/MyDrive/TeraiNet/samples/lila_bc_image_urls_wild_boar.csv.




In [ ]:
species_samples_dict = {
    'wild boar': 3100,
}

sampled_image_urls = sample_n_images_per_species(samples_file, species_samples_dict, column_to_filter)
!rm "$samples_file"
sampled_image_urls['class_number'] = class_number
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_samples_file = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.to_csv(sampled_samples_file, index=False, sep=',')
!echo "Sampled rows: $(tail -n +2 "$sampled_samples_file" | wc -l)"
sampled_image_urls[column_to_filter].value_counts()

Sampled rows: 3100


,count
common_name,
wild boar,3100


### Buffalo

Use `african buffalo,cape buffalo` because camera trap images of gaur are unavailable.

In [ ]:
class_name = 'buffalo'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = 'african buffalo,cape buffalo'
samples_file = get_image_urls(samples_dir, class_name, get_image_urls_script, urls_and_labels, column_to_filter, values_to_filter)

Getting image URLs based on common_name equal to any of {african buffalo,cape buffalo}...
Total matching rows found: 94114
Filtering complete. Result written to drive/MyDrive/TeraiNet/samples/lila_bc_image_urls_buffalo.csv.




In [ ]:
species_samples_dict = {
    'african buffalo': 1550,
    'cape buffalo': 1550,
}

sampled_image_urls = sample_n_images_per_species(samples_file, species_samples_dict, column_to_filter)
!rm "$samples_file"
sampled_image_urls['class_number'] = class_number
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_samples_file = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.to_csv(sampled_samples_file, index=False, sep=',')
!echo "Sampled rows: $(tail -n +2 "$sampled_samples_file" | wc -l)"
sampled_image_urls[column_to_filter].value_counts()

Sampled rows: 3100


,count
common_name,
african buffalo,1550
cape buffalo,1550


### Rhino

Use `white rhinoceros` because camera trap images of Indian rhinoceros are unavailable.

In [ ]:
class_name = 'rhino'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = 'white rhinoceros'
samples_file = get_image_urls(samples_dir, class_name, get_image_urls_script, urls_and_labels, column_to_filter, values_to_filter)

Getting image URLs based on common_name equal to any of {white rhinoceros}...
Total matching rows found: 7307
Filtering complete. Result written to drive/MyDrive/TeraiNet/samples/lila_bc_image_urls_rhino.csv.




In [ ]:
species_samples_dict = {
    'white rhinoceros': 3100,
}

sampled_image_urls = sample_n_images_per_species(samples_file, species_samples_dict, column_to_filter)
!rm "$samples_file"
sampled_image_urls['class_number'] = class_number
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_samples_file = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.to_csv(sampled_samples_file, index=False, sep=',')
!echo "Sampled rows: $(tail -n +2 "$sampled_samples_file" | wc -l)"
sampled_image_urls[column_to_filter].value_counts()

Sampled rows: 3100


,count
common_name,
white rhinoceros,3100


### Elephant

Include `african elephant,african bush elephant` because there are not enough camera trap images of Asian elephants.

In [ ]:
class_name = 'elephant'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = 'asian elephant,african elephant,african bush elephant'
samples_file = get_image_urls(samples_dir, class_name, get_image_urls_script, urls_and_labels, column_to_filter, values_to_filter)

Getting image URLs based on common_name equal to any of {asian elephant,african elephant,african bush elephant}...
Total matching rows found: 188879
Filtering complete. Result written to drive/MyDrive/TeraiNet/samples/lila_bc_image_urls_elephant.csv.




In [ ]:
species_samples_dict = {
    'asian elephant': 325,
    'african elephant': 1388,
    'african bush elephant': 1388,
}

sampled_image_urls = sample_n_images_per_species(samples_file, species_samples_dict, column_to_filter)
!rm "$samples_file"
sampled_image_urls['class_number'] = class_number
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_samples_file = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.to_csv(sampled_samples_file, index=False, sep=',')
!echo "Sampled rows: $(tail -n +2 "$sampled_samples_file" | wc -l)"
sampled_image_urls[column_to_filter].value_counts()

Sampled rows: 3101


,count
common_name,
african elephant,1388
african bush elephant,1388
asian elephant,325


### Bird

In [ ]:
class_name = 'bird'
class_number = config['classes'][class_name]
column_to_filter = 'common_name'
values_to_filter = 'bird'
samples_file = get_image_urls(samples_dir, class_name, get_image_urls_script, urls_and_labels, column_to_filter, values_to_filter)

Getting image URLs based on common_name equal to any of {bird}...
Total matching rows found: 297686
Filtering complete. Result written to drive/MyDrive/TeraiNet/samples/lila_bc_image_urls_bird.csv.




In [ ]:
species_samples_dict = {
    'bird': 3100,
}

sampled_image_urls = sample_n_images_per_species(samples_file, species_samples_dict, column_to_filter)
!rm "$samples_file"
sampled_image_urls['class_number'] = class_number
sampled_image_urls = add_subset_column(sampled_image_urls, train_ratio)
sampled_samples_file = os.path.join(samples_dir, 'lila_bc_image_urls_' + class_name + '_sampled_class_number_' + str(class_number) + '.csv')
sampled_image_urls.to_csv(sampled_samples_file, index=False, sep=',')
!echo "Sampled rows: $(tail -n +2 "$sampled_samples_file" | wc -l)"
sampled_image_urls[column_to_filter].value_counts()

Sampled rows: 3100


,count
common_name,
bird,3100
